# Power / Minimum-Detectable-Effect Analysis for RQ1

**Owner:** Jici · **Depends on:** RQ1 (`rq1_methodology.md`), Option B labels

**Motivation.** RQ1 found SPD = +0.029 with a 95% bootstrap CI [-0.005, +0.065]
that crosses zero -> "not statistically significant." That phrase is ambiguous:
it could mean *no real effect*, or it could mean *our sample (590 labeled slots,
100 queries) is too small to detect a real small effect*. Prof. Sushmita's
suggestion to try different ranking cutoffs was, implicitly, a question about
this same ambiguity. This notebook answers it directly and quantitatively:
**given our actual sample size and query-clustering structure, how large would
the true SPD need to be for our bootstrap test to reliably detect it?**

**Method summary.**
1. Reconstruct each of the 100 neutral queries' real `(n_q, e_q)` -- labeled
   slot count and elite count -- from the actual Option B retrieval data.
2. Validate: re-running RQ1's exact query-resampling bootstrap on this
   reconstruction must reproduce the published SPD/CI.
3. For a grid of *synthetic* true SPD values, simulate `e_q_sim ~
   Binomial(n_q, 0.144 + target_SPD)` per query (keeping each query's real
   `n_q`, so real coverage/clustering heterogeneity across queries is
   preserved), then run the identical bootstrap on the simulated data and
   record whether the CI excludes 0. Repeating this many times per target SPD
   gives empirical **power** = P(detect | true SPD = target).
4. Calibration check: power at target SPD = 0 should be ~5% (the nominal
   false-positive rate) -- this validates the simulation pipeline.
5. Read off the **minimum detectable effect (MDE) at 80% power**, and compare
   it to the observed point estimate (+0.029).

**Key limitation of this design (disclosed upfront):** the simulation draws
each query's synthetic elite count independently, `e_q_sim ~ Binomial(n_q, p)`.
The real data likely has some between-query correlation in elite rate (e.g.
certain subcategories draw from certain institutions more), which the
real bootstrap-over-queries analysis correctly accounts for when analyzing
*observed* data, but which this independent-per-query generative model does
not reproduce when *simulating new data*. This is a standard simplification
in bootstrap power analysis; its likely direction is to make the power
estimates mildly **optimistic** (a bit higher than the true power would be,
since it removes one source of extra variance), so the MDE reported here
should be read as a **lower bound** on the true MDE.

In [ ]:
import json, os, glob
import numpy as np
import matplotlib.pyplot as plt


## 0. Config

In [ ]:
BASELINE_SHARE = 0.1438     # Option B baseline elite share (438/1000 found), locked convention
N_BOOT_VALIDATE = 5000      # matches RQ1's primary bootstrap (rq1_methodology.md §5)
N_BOOT_POWER    = 3000      # inner bootstrap reps per simulated dataset (reduced for speed; see §6)
R_REPLICATES    = 2000      # outer simulation replicates per target SPD
TARGET_SPDS     = [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15]
OBSERVED_SPD    = 0.0290    # RQ1 published point estimate, for reference on the plot
POWER_TARGET    = 0.80

def find(hint):
    for base in ['data', '.', '..']:
        hits = glob.glob(f'{base}/**/{hint}', recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(hint)


## 1. Reconstruct real per-query (n_q, e_q)

Same neutral-query scope (q001-q100) and same Option B labels as RQ1
(`sample_labels_1000.json` + `retrieval_labels.json` merged; `elite_label` is
present only for `coverage == 'found'` papers, matching the locked convention
that unlabeled papers are excluded rather than treated as non-elite).

In [ ]:
with open(find('retrieval_results.json')) as f:
    retrieval = json.load(f)
queries = [q for q in retrieval if q.get('type', 'neutral') == 'neutral']
queries = [q for q in queries if q['query_id'] <= 'q100']
print(f'neutral queries: {len(queries)}')

label_of = {}
for fn in ['sample_labels_1000.json', 'retrieval_labels.json']:
    with open(find(fn)) as f:
        for r in json.load(f)['labels']:
            label_of[str(r['paper_id'])] = r.get('elite_label')  # None if not found/no_affiliation

n_list, e_list = [], []
for q in queries:
    labs = [label_of.get(pid) for pid in q['retrieved_paper_ids']]
    labs = [x for x in labs if x is not None]
    n_list.append(len(labs))
    e_list.append(sum(labs))
n_arr = np.array(n_list)
e_arr_real = np.array(e_list)
Q = len(n_arr)
print(f'total labeled slots: {n_arr.sum()} | total elite: {e_arr_real.sum()} | '
      f'retrieved elite share: {e_arr_real.sum()/n_arr.sum():.4f}')

## 2. Validate: reproduce RQ1's published SPD and bootstrap CI

This must match `rq1_methodology.md` §6 (SPD +0.0290, CI [-0.0054, +0.0648])
before trusting anything downstream.

In [ ]:
def bootstrap_ci(n_arr, e_arr, baseline, n_boot, rng):
    """Query-resampling bootstrap for SPD, vectorized. Resamples QUERY indices
    (not individual papers) with replacement -- preserves within-query
    correlation, exactly mirroring RQ1's methodology (§5)."""
    Qn = len(n_arr)
    idx = rng.integers(0, Qn, size=(n_boot, Qn))
    n_tot = n_arr[idx].sum(axis=1)
    e_tot = e_arr[idx].sum(axis=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        spds = e_tot / n_tot - baseline
    return np.nanpercentile(spds, [2.5, 97.5])

rng = np.random.default_rng(42)
spd_point = e_arr_real.sum() / n_arr.sum() - BASELINE_SHARE
ci_lo, ci_hi = bootstrap_ci(n_arr, e_arr_real, BASELINE_SHARE, N_BOOT_VALIDATE, rng)
print(f'SPD point estimate : {spd_point:+.4f}   (RQ1 reports +0.0290)')
print(f'Bootstrap 95% CI   : [{ci_lo:+.4f}, {ci_hi:+.4f}]   (RQ1 reports [-0.0054, +0.0648])')
assert abs(spd_point - 0.0290) < 0.001, 'SPD point estimate does not match RQ1 -- check data/labels'
print('\nMATCH CONFIRMED -- per-query reconstruction and bootstrap procedure are correct.')

## 3. Power simulation

For each target SPD, simulate many synthetic datasets (same real `n_q` per
query, elite counts redrawn at rate `0.1438 + target_SPD`), run the identical
bootstrap on each, and record the fraction whose CI excludes 0.

In [ ]:
def power_at(target_spd, n_arr, baseline, R, n_boot, rng):
    Qn = len(n_arr)
    p_sim = baseline + target_spd
    e_sims = rng.binomial(n_arr[None, :], p_sim, size=(R, Qn))  # (R, Q) synthetic elite counts
    detected = 0
    for r in range(R):
        lo, hi = bootstrap_ci(n_arr, e_sims[r], baseline, n_boot, rng)
        if lo > 0 or hi < 0:
            detected += 1
    return detected / R

power_curve = {}
for spd in TARGET_SPDS:
    rng_spd = np.random.default_rng(int(round(spd * 10000)) + 1)
    pw = power_at(spd, n_arr, BASELINE_SHARE, R_REPLICATES, N_BOOT_POWER, rng_spd)
    power_curve[spd] = pw
    print(f'target SPD = {spd:+.2f}: power = {pw:.3f}')

## 4. Calibration check (Type-I error)

At `target_SPD = 0` (truly no effect), the false-positive rate should be close
to the nominal 5%. If it is, the simulation pipeline is validated and the
power numbers above can be trusted.

In [ ]:
type1 = power_curve[0.0]
print(f'Type-I error at target SPD=0: {type1:.3f}  (nominal alpha=0.05)')
if abs(type1 - 0.05) < 0.03:
    print('-> within simulation noise of the nominal rate; pipeline is calibrated.')
else:
    print('-> WARNING: check n_boot / R_REPLICATES, or the bootstrap implementation.')

## 5. Minimum detectable effect (MDE) at 80% power

Linear interpolation on the power curve to find the target SPD at which power
crosses 0.80.

In [ ]:
spds_sorted = np.array(sorted(power_curve.keys()))
powers_sorted = np.array([power_curve[s] for s in spds_sorted])

if (powers_sorted >= POWER_TARGET).any():
    i = np.argmax(powers_sorted >= POWER_TARGET)
    if i == 0:
        mde_80 = spds_sorted[0]
    else:
        x0, x1 = spds_sorted[i-1], spds_sorted[i]
        y0, y1 = powers_sorted[i-1], powers_sorted[i]
        mde_80 = x0 + (POWER_TARGET - y0) * (x1 - x0) / (y1 - y0)
else:
    mde_80 = float('nan')

power_at_observed = np.interp(OBSERVED_SPD, spds_sorted, powers_sorted)
print(f'MDE at 80% power        : SPD ~= {mde_80:.4f}')
print(f'Observed point estimate : SPD  = {OBSERVED_SPD:+.4f}')
print(f'Power AT the observed point estimate (if it were the true effect): {power_at_observed:.3f}')

## 6. Quick analytic cross-check (independence-assumption lower bound)

A closed-form normal-approximation MDE, **ignoring query clustering**
(treating all 590 labeled slots as independent Bernoulli draws). This should
be somewhat **smaller** than the simulation-based MDE above, since ignoring
clustering understates the true variance -- if it isn't smaller, that's a red
flag worth investigating.

In [ ]:
from scipy import stats
n_total = int(n_arr.sum())
z_alpha = stats.norm.ppf(0.975)   # two-sided alpha=0.05
z_beta  = stats.norm.ppf(POWER_TARGET)
se_naive = np.sqrt(BASELINE_SHARE * (1 - BASELINE_SHARE) / n_total)
mde_analytic = (z_alpha + z_beta) * se_naive
print(f'Naive (independence-assumed) MDE @ 80% power: SPD ~= {mde_analytic:.4f}')
print(f'Simulation-based MDE (accounts for query clustering): SPD ~= {mde_80:.4f}')
print('Simulation MDE should be >= analytic MDE (clustering inflates variance).')

## 7. Power curve plot

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(spds_sorted, powers_sorted, 'o-', color='#2e5fa3', label='Empirical power (this notebook)')
ax.axhline(POWER_TARGET, ls='--', c='#888', lw=1, label='80% power target')
ax.axhline(0.05, ls=':', c='#bbb', lw=1, label='5% (Type-I reference)')
ax.axvline(OBSERVED_SPD, ls='-.', c='#c0392b', lw=1.3,
           label=f'Observed RQ1 point estimate ({OBSERVED_SPD:+.3f})')
if not np.isnan(mde_80):
    ax.plot([mde_80], [POWER_TARGET], 'D', color='#c0392b', markersize=7)
    ax.annotate(f'MDE@80% \u2248 {mde_80:.3f}', (mde_80, POWER_TARGET),
                textcoords='offset points', xytext=(8, -14), fontsize=9)
ax.set_xlabel('Target (synthetic) true SPD')
ax.set_ylabel('Power = P(bootstrap CI excludes 0)')
ax.set_title(f'RQ1 statistical power vs. true SPD\n(n={Q} queries, {int(n_arr.sum())} labeled slots)')
ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('rq1_power_curve.png', dpi=120)
plt.show()

## 8. Save results

In [ ]:
out = {
    'baseline_share': BASELINE_SHARE,
    'n_queries': int(Q),
    'n_labeled_slots': int(n_arr.sum()),
    'observed_spd': OBSERVED_SPD,
    'power_curve': {str(k): round(float(v), 4) for k, v in power_curve.items()},
    'type1_error_at_spd0': round(float(type1), 4),
    'mde_at_80pct_power': round(float(mde_80), 4) if not np.isnan(mde_80) else None,
    'power_at_observed_spd': round(float(power_at_observed), 4),
    'analytic_mde_naive_independence': round(float(mde_analytic), 4),
    'config': {
        'n_boot_power': N_BOOT_POWER, 'r_replicates': R_REPLICATES,
        'target_spds': TARGET_SPDS,
    },
}
os.makedirs('results', exist_ok=True)
with open('results/rq1_power_analysis.json', 'w') as f:
    json.dump(out, f, indent=2)
print('saved results/rq1_power_analysis.json')
print(json.dumps(out, indent=2))

## 9. Reading the result (for the report / methodology doc)

Fill in after running: e.g. *"At our sample size (100 queries, ~590 labeled
slots), the bootstrap test has 80% power to detect a true SPD of ~X. Our
observed point estimate (+0.029) would, if it were the true effect, be
detected only ~Y% of the time -- so the non-significant result is consistent
with either a true near-zero effect, or an undetected small effect of this
magnitude. We can rule out, with high confidence, any elite-retrieval tilt
larger than ~X."*

**Limitations (carry into the report):**
1. The simulation model draws each query's synthetic elite count
   independently at a fixed rate; real data likely has some between-query
   correlation (topic-institution association), which this model does not
   reproduce. The reported MDE is a **lower bound** (true MDE is likely
   somewhat higher / true power somewhat lower than shown here).
2. This analysis targets **RQ1 only**. The same method (query-resampling
   bootstrap + per-query synthetic-effect injection) extends naturally to
   RQ2-A (institutional citation amplification) and RQ3 (SPD improvement
   under re-ranking), but has not been run there.
3. `N_BOOT_POWER` (3000) is lower than RQ1's primary bootstrap (5000) purely
   for simulation speed across `R_REPLICATES` x `len(TARGET_SPDS)` runs; §2
   already validated that the bootstrap procedure itself reproduces RQ1's
   published CI at `N_BOOT_VALIDATE=5000`.